# 01 · Fuentes de datos

**Objetivo del proyecto:** predecir si un juego de Steam que aún no ha salido va a ser **exitoso**.

En este notebook:
1. Cargamos el **dataset histórico** de Kaggle (*Steam Games Dataset – daily updates*, ~150k juegos).
2. Cargamos los **próximos lanzamientos** descargados de la API de cleanedweb (*Steam Upcoming Games API*).
3. **Enriquecemos** los próximos lanzamientos con la API pública de la tienda de Steam (`appdetails`), porque la API de cleanedweb solo trae título, plataformas, fecha, precio y tags.
4. Comparamos columnas: el modelo solo puede usar variables **disponibles antes del lanzamiento** y **presentes en ambas fuentes**.

Las salidas se guardan en `data/raw/`.

In [3]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_columns", 100)

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)
sorted(p.name for p in RAW.iterdir())

['steam_games.csv',
 'steam_games_reviews.csv',
 'steam_upcoming_api.json',
 'steam_upcoming_games_sample.csv',
 'steam_upcoming_games_sample.jsonl',
 'upcoming_appdetails.jsonl',
 'upcoming_enriched.json']

## 1. Dataset histórico (Kaggle)

In [4]:
HIST_PATH = RAW / "steam_games.csv" 

df_hist = pd.read_csv(HIST_PATH, low_memory=False)
df_hist = df_hist.rename(columns={"app_id": "appid"})
print(df_hist.shape)
df_hist.columns.tolist()

(150863, 42)


['appid',
 'name',
 'release_date',
 'price',
 'price_status',
 'estimated_owners',
 'developers',
 'publishers',
 'genres',
 'categories',
 'tags',
 'positive',
 'negative',
 'recommendations',
 'peak_ccu',
 'metacritic_score',
 'user_score',
 'average_playtime_forever',
 'median_playtime_forever',
 'average_playtime_2weeks',
 'median_playtime_2weeks',
 'short_description',
 'about_the_game',
 'detailed_description',
 'notes',
 'achievements',
 'dlc_count',
 'packages',
 'supported_languages',
 'full_audio_languages',
 'windows',
 'mac',
 'linux',
 'header_image',
 'screenshots',
 'movies',
 'website',
 'support_url',
 'support_email',
 'metacritic_url',
 'steam_store_available',
 'steam_spy_available']

In [5]:
df_hist.head(3).T

,0,1,2
appid,2582880,3508520,1508190
name,Northrealm Tribes,Echoes of the Abyss,Finding Words
release_date,2026-10-20,2026-10-21,2026-10-21
price,0.0,0.99,3.99
price_status,free,paid,paid
estimated_owners,"0 .. 20,000","0 .. 20,000","0 .. 20,000"
developers,"[""DevSanja""]","[""lijj1235666""]","[""Odd Build""]"
publishers,"[""DevSanja""]","[""lij1235666""]","[""Odd Build""]"
genres,"[{""id"": ""4"", ""description"": ""Casual""}, {""id"": ...","[{""id"": ""1"", ""description"": ""Action""}, {""id"": ...","[{""id"": ""4"", ""description"": ""Casual""}]"
categories,"[{""id"": 2, ""description"": ""Single-player""}, {""...","[{""id"": 2, ""description"": ""Single-player""}, {""...","[{""id"": 2, ""description"": ""Single-player""}, {""..."


In [6]:
resumen = pd.DataFrame({
    "dtype": df_hist.dtypes.astype(str),
    "pct_nulos": (df_hist.isna().mean() * 100).round(1),
    "n_unicos": df_hist.nunique(),
})
resumen

,dtype,pct_nulos,n_unicos
appid,int64,0.0,150863
name,str,0.0,149284
release_date,str,0.1,5401
price,float64,0.9,991
price_status,str,0.0,3
estimated_owners,str,0.0,26
developers,str,0.0,90162
publishers,str,0.0,81298
genres,str,0.0,4337
categories,str,0.0,25279


### 1.1 Distribución por año de lanzamiento

Para entrenar solo nos sirven juegos que lleven **al menos 12 meses** a la venta (si no, aún no han acumulado reseñas y parecerían fracasos aunque no lo sean).

In [7]:
fechas = pd.to_datetime(df_hist["release_date"], errors="coerce")
print("Fechas no parseables:", fechas.isna().sum())
fechas.dt.year.value_counts().sort_index().tail(15)

Fechas no parseables: 82


release_date
2012.0      322
2013.0      467
2014.0     1521
2015.0     2513
2016.0     4137
2017.0     5918
2018.0     7449
2019.0     7232
2020.0     8788
2021.0    11043
2022.0    12253
2023.0    14538
2024.0    19898
2025.0    24688
2026.0    28835
Name: count, dtype: int64

### 1.2 Columnas pre‑lanzamiento vs post‑lanzamiento

Las columnas **post‑lanzamiento** (reseñas, propietarios, horas de juego…) **no pueden ser features**: un juego que aún no ha salido no las tiene. Algunas sí servirán para **construir el target**.

In [8]:
# Solo existen DESPUÉS del lanzamiento -> NUNCA features (algunas sirven para el target)
POST_LAUNCH = [
    "positive", "negative", "recommendations", "estimated_owners", "peak_ccu",
    "metacritic_score", "metacritic_url", "user_score",
    "average_playtime_forever", "median_playtime_forever",
    "average_playtime_2weeks", "median_playtime_2weeks",
    "dlc_count",
]

# Sin valor predictivo o de uso técnico -> se descartan
DESCARTAR = [
    "header_image", "website", "support_url", "support_email",
    "packages", "notes", "price_status",
    "steam_store_available", "steam_spy_available",   # se usan para filtrar, no como feature
]

pre_candidatas = [c for c in df_hist.columns if c not in POST_LAUNCH + DESCARTAR]

print("Post-lanzamiento (posibles para el target):\n", POST_LAUNCH)
print("\nDescartadas:\n", DESCARTAR)
print("\nCandidatas a feature:\n", pre_candidatas)
print("\nCobertura SteamSpy:\n", df_hist["steam_spy_available"].value_counts(dropna=False))
print("\nprice_status:\n", df_hist["price_status"].value_counts(dropna=False))

Post-lanzamiento (posibles para el target):
 ['positive', 'negative', 'recommendations', 'estimated_owners', 'peak_ccu', 'metacritic_score', 'metacritic_url', 'user_score', 'average_playtime_forever', 'median_playtime_forever', 'average_playtime_2weeks', 'median_playtime_2weeks', 'dlc_count']

Descartadas:
 ['header_image', 'website', 'support_url', 'support_email', 'packages', 'notes', 'price_status', 'steam_store_available', 'steam_spy_available']

Candidatas a feature:
 ['appid', 'name', 'release_date', 'price', 'developers', 'publishers', 'genres', 'categories', 'tags', 'short_description', 'about_the_game', 'detailed_description', 'achievements', 'supported_languages', 'full_audio_languages', 'windows', 'mac', 'linux', 'screenshots', 'movies']

Cobertura SteamSpy:
 steam_spy_available
True    150863
Name: count, dtype: int64

price_status:
 price_status
paid           120702
free            28732
unavailable      1429
Name: count, dtype: int64


## 2. Próximos lanzamientos (API cleanedweb)

In [9]:
def load_upcoming(path: Path) -> pd.DataFrame:
    """Carga el fichero descargado de la API (json o csv)."""
    if path.suffix == ".json":
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):  # por si viene envuelto: {"items": [...]}
            for k in ("items", "data", "records", "results"):
                if k in data:
                    data = data[k]
                    break
        return pd.json_normalize(data)
    return pd.read_csv(path)


UP_PATH = RAW / "steam_upcoming_api.json"

df_up = load_upcoming(UP_PATH)
print(df_up.shape)
df_up.columns.tolist()

(36, 22)


['alt',
 'ds_appid',
 'ds_crtrids',
 'ds_itemkey',
 'ds_tagids',
 'image',
 'platforms',
 'release_date',
 'tab_item_name',
 'tags',
 'title',
 'url',
 'ds_descids',
 'vr_supported',
 'bundlediscount',
 'discount',
 'discount_aria_label',
 'discount_original_price',
 'discount_pct',
 'prepurchase_tag',
 'price_final',
 'tooltip_text']

In [10]:
print("HIST:", [c for c in df_hist.columns if "app" in c.lower() or "id" in c.lower()])
print("UP:  ", [c for c in df_up.columns if "app" in c.lower() or "id" in c.lower()])

HIST: ['appid']
UP:   ['ds_appid', 'ds_crtrids', 'ds_tagids', 'ds_descids']


In [11]:
df_up = df_up.rename(columns={"ds_appid": "appid"})
df_up["appid"] = pd.to_numeric(df_up["appid"], errors="coerce").astype("Int64")
df_up = df_up.dropna(subset=["appid"]).drop_duplicates("appid").reset_index(drop=True)

ya_en_hist = df_up["appid"].isin(df_hist["appid"])
print("Próximos lanzamientos únicos:", len(df_up))
print("De ellos ya aparecen en el histórico:", ya_en_hist.sum())
df_up.head()

Próximos lanzamientos únicos: 36
De ellos ya aparecen en el histórico: 0


,alt,appid,ds_crtrids,ds_itemkey,ds_tagids,image,platforms,release_date,tab_item_name,tags,title,url,ds_descids,vr_supported,bundlediscount,discount,discount_aria_label,discount_original_price,discount_pct,prepurchase_tag,price_final,tooltip_text
0,No Asteroids,5527800,[46720472],App_5527800,"[560542, 5981, 1685, 5363, 1755, 3843, 35079]",https://shared.akamai.steamstatic.com/store_it...,"[win, mac]","Nov 24, 2026",No Asteroids,"[Incremental, Mining, Co-op, Destruction]","No Asteroids Incremental, Mining, Co-op, Dest...",https://store.steampowered.com/app/5527800/No_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Am I Nima,3224600,"[45444638, 44870581]",App_3224600,"[1721, 4486, 11014, 6815, 3799, 1664, 7208]",https://shared.akamai.steamstatic.com/store_it...,[win],"Nov 27, 2026",Am I Nima,"[Psychological Horror, Choose Your Own Adventu...","Am I Nima Psychological Horror, Choose Your O...",https://store.steampowered.com/app/3224600/Am_...,[5],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,No Body No Problem!,4954970,"[44123520, 45941845]",App_4954970,"[19, 597, 4106, 1773, 7178, 3834, 4191]",https://shared.akamai.steamstatic.com/store_it...,[win],"Nov 26, 2026",No Body No Problem!,"[Action, Casual, Action-Adventure, Arcade]","No Body No Problem! Action, Casual, Action-Ad...",https://store.steampowered.com/app/4954970/No_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Winx Club - Moonlight Quest,5189950,[46170095],App_5189950,"[1695, 7208, 4057, 5350, 1684, 21, 3834]",https://shared.akamai.steamstatic.com/store_it...,[win],"Nov 29, 2026",Winx Club - Moonlight Quest,"[Open World, Female Protagonist, Magic, Family...","Winx Club - Moonlight Quest Open World, Femal...",https://store.steampowered.com/app/5189950/Win...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Arctic Drive,4009490,[37611909],App_4009490,"[493, 1644, 1662, 1685, 1100687, 3843, 3968]",https://shared.akamai.steamstatic.com/store_it...,[win],"Nov 25, 2026",Arctic Drive,"[Early Access, Driving, Survival, Co-op]","Arctic Drive Early Access, Driving, Survival,...",https://store.steampowered.com/app/4009490/Arc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
print(df_up["price_final"].notna().sum(), "de", len(df_up), "tienen price_final")
print(df_up[["title", "price_final"]].dropna().head(15))

1 de 36 tienen price_final
                                                title price_final
10  Warhammer 40,000: Dawn of War IV  Strategy, RT...        5399


## 3. Enriquecimiento con la Steam Store API (`appdetails`)

La API de próximos lanzamientos no trae desarrollador, géneros, categorías, idiomas, capturas, etc.
La tienda de Steam tiene un endpoint público (sin API key) que sí los devuelve también para juegos no lanzados:

`https://store.steampowered.com/api/appdetails?appids=<appid>&cc=us&l=english`

- `cc=us` → precio en **USD**, como en el dataset de Kaggle.
- `l=english` → géneros y categorías en **inglés**, para que coincidan con Kaggle.
- Límite aproximado: **~200 peticiones cada 5 min** → esperamos 1,6 s entre llamadas.
- Se guarda cada respuesta en `upcoming_appdetails.jsonl`: si se corta, al relanzar la celda **continúa donde lo dejó**.

In [13]:
CACHE = RAW / "upcoming_appdetails.jsonl"
URL = "https://store.steampowered.com/api/appdetails"


def fetch_appdetails(appid: int, session: requests.Session):
    r = session.get(URL, params={"appids": appid, "cc": "us", "l": "english"}, timeout=20)
    if r.status_code == 429:
        return "RATE_LIMIT"
    r.raise_for_status()
    entry = (r.json() or {}).get(str(appid), {})
    return entry.get("data") if entry.get("success") else None


done = set()
if CACHE.exists():
    with open(CACHE, encoding="utf-8") as f:
        done = {json.loads(line)["appid"] for line in f}

pending = [int(a) for a in df_up["appid"] if int(a) not in done]
print(f"Ya descargados: {len(done)} | Pendientes: {len(pending)} "
      f"(~{len(pending) * 1.6 / 60:.0f} min)")

Ya descargados: 36 | Pendientes: 0 (~0 min)


In [14]:
session = requests.Session()
with open(CACHE, "a", encoding="utf-8") as f:
    for i, appid in enumerate(pending, 1):
        data = fetch_appdetails(appid, session)
        while data == "RATE_LIMIT":
            print("429 (demasiadas peticiones) -> espero 5 min")
            time.sleep(300)
            data = fetch_appdetails(appid, session)
        f.write(json.dumps({"appid": appid, "data": data}, ensure_ascii=False) + "\n")
        f.flush()
        if i % 50 == 0:
            print(f"{i}/{len(pending)}")
        time.sleep(1.6)
print("Terminado")

Terminado


In [15]:
def flatten(appid: int, d: dict | None) -> dict:
    """Convierte la respuesta de appdetails en una fila con nombres parecidos a Kaggle."""
    if not d:
        return {"appid": appid}
    po = d.get("price_overview") or {}
    pf = d.get("platforms") or {}
    rd = d.get("release_date") or {}
    if po:
        price = po.get("initial", 0) / 100
    else:
        price = 0.0 if d.get("is_free") else None   # None = precio aún no anunciado
    return {
        "appid": appid,
        "name": d.get("name"),
        "type": d.get("type"),
        "release_date_store": rd.get("date"),
        "coming_soon": rd.get("coming_soon"),
        "required_age": d.get("required_age"),
        "is_free": d.get("is_free"),
        "price": price,
        "windows": pf.get("windows"),
        "mac": pf.get("mac"),
        "linux": pf.get("linux"),
        "developers": d.get("developers", []),
        "publishers": d.get("publishers", []),
        "genres": [g["description"] for g in d.get("genres", [])],
        "categories": [c["description"] for c in d.get("categories", [])],
        "supported_languages": d.get("supported_languages"),
        "achievements": (d.get("achievements") or {}).get("total", 0),
        "n_screenshots": len(d.get("screenshots", [])),
        "n_movies": len(d.get("movies", [])),
        "short_description": d.get("short_description"),
        "detailed_description": d.get("detailed_description"),
        "header_image": d.get("header_image"),
    }


with open(CACHE, encoding="utf-8") as f:
    rows = [flatten(r["appid"], r["data"]) for r in map(json.loads, f)]
df_details = pd.DataFrame(rows)

print("Sin datos en la tienda:", df_details["name"].isna().sum())
df_details["type"].value_counts(dropna=False)

Sin datos en la tienda: 0


type
game    36
Name: count, dtype: int64

In [16]:
cols_api = [c for c in ["appid", "title", "tags", "vr_supported", "release_date", "price_final"]
            if c in df_up.columns]

df_up_full = (
    df_details[df_details["type"] == "game"]                 # fuera DLCs, bandas sonoras, demos...
    .merge(df_up[cols_api].rename(columns={"release_date": "release_date_api"}),
           on="appid", how="left")
    .reset_index(drop=True)
)
print(df_up_full.shape)
df_up_full.head(3).T

(36, 27)


,0,1,2
appid,5527800,3224600,4954970
name,No Asteroids,Am I Nima,No Body No Problem!
type,game,game,game
release_date_store,"Nov 24, 2026","Nov 27, 2026","Nov 26, 2026"
coming_soon,False,True,True
required_age,0,0,0
is_free,False,False,False
price,7.99,NaN,NaN
windows,True,True,True
mac,True,False,False


In [17]:
OUT = RAW / "upcoming_enriched.json"
df_up_full.to_json(OUT, orient="records", force_ascii=False, indent=1)
print("Guardado en", OUT)

Guardado en ..\data\raw\upcoming_enriched.json


## 4. Comparación de columnas entre fuentes

Las features finales tienen que existir (o poder construirse) en **las dos** fuentes.

In [18]:
comunes = sorted(set(df_hist.columns) & set(df_up_full.columns))
solo_hist = sorted(set(pre_candidatas) - set(df_up_full.columns))
solo_up = sorted(set(df_up_full.columns) - set(df_hist.columns))

print("Comunes:\n", comunes)
print("\nSolo en histórico (pre-lanzamiento):\n", solo_hist)
print("\nSolo en próximos:\n", solo_up)

Comunes:
 ['achievements', 'appid', 'categories', 'detailed_description', 'developers', 'genres', 'header_image', 'linux', 'mac', 'name', 'price', 'publishers', 'short_description', 'supported_languages', 'tags', 'windows']

Solo en histórico (pre-lanzamiento):
 ['about_the_game', 'full_audio_languages', 'movies', 'release_date', 'screenshots']

Solo en próximos:
 ['coming_soon', 'is_free', 'n_movies', 'n_screenshots', 'price_final', 'release_date_api', 'release_date_store', 'required_age', 'title', 'type', 'vr_supported']


In [19]:
# Ejemplo de formato de las columnas de listas en cada fuente (para unificarlas en el 02)
for col in ["genres", "categories", "tags", "developers", "supported_languages", "screenshots"]:
    if col in df_hist.columns:
        print(f"[HIST] {col}: {str(df_hist[col].dropna().iloc[0])[:150]}")
    if col in df_up_full.columns:
        print(f"[UP]   {col}: {str(df_up_full[col].dropna().iloc[0])[:150]}")
    print()

[HIST] genres: [{"id": "4", "description": "Casual"}, {"id": "23", "description": "Indie"}]
[UP]   genres: ['Casual', 'Indie', 'Simulation']

[HIST] categories: [{"id": 2, "description": "Single-player"}, {"id": 1, "description": "Multi-player"}, {"id": 49, "description": "PvP"}, {"id": 36, "description": "Onl
[UP]   categories: ['Single-player', 'Multi-player', 'Co-op', 'Online Co-op', 'Cross-Platform Multiplayer', 'Steam Achievements', 'Full controller support', 'Camera Comf

[HIST] tags: []
[UP]   tags: ['Incremental', 'Mining', 'Co-op', 'Destruction']

[HIST] developers: ["DevSanja"]
[UP]   developers: ['Unruled Games']

[HIST] supported_languages: English, German, Russian
[UP]   supported_languages: English, German, Japanese, Portuguese - Brazil, Russian, Simplified Chinese, Spanish - Latin America, Turkish, Ukrainian, Arabic, French, Polish, Ital

[HIST] screenshots: [{"id": 0, "path_thumbnail": "https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2582880/0c18bd02a

## 5. Conclusiones

- Nº de juegos en el histórico y rango de fechas: 150.863 juegos x 42 columnas. El volumen de lanzamientos crece cada año (2.513 en 2015 -> 24.688 en 2025 -> 28.835 en 2026), lo que refleja la saturación de Steam. Solo 82 fechas no son legibles
- Nº de próximos lanzamientos válidos (tipo `game`): 36 juegos de la API de cleanedweb, ninguno presente en el histórico. La API apenas trae datos útiles, por lo que se enriquecieron con la API de la tienda de Steam (appdetails): 36/36 fichas completas, todas de tipo 'game'.
- Variables comunes que usaremos como features: precio, plataformas, géneros, categorías, tags, idiomas, logros, nº de capturas y vídeos, longitud de las descripciones, fecha, desarrollador y publisher.
- Variables post‑lanzamiento que usaremos para definir el target: positive, negative, recommendations, estimated_owners, peak_ccu, metacritic_*, user_score, tiempos de juego y dlc_count.
- Diferencias de formato a unificar en `02_LimpiezaEDA.ipynb`: géneros y categorías vienen como JSON con 'description' en Kaggle y como listas en la API; los idiomas de la tienda incluyen HTML; capturas y vídeos hay que contarlos en el histórico.
- Limitacion: muchos juegos próximos no tienen precio anunciado (Steam no lo publica hasta el lanzamiento) y price_final solo existe en 1 de 36 (además es un precio de reserva con descuento), por lo que no se utiliza.

## 6. Versión ligera del histórico para el repositorio

In [20]:
import sys
sys.path.insert(0, str(Path("../src").resolve()))
from data_processing import count_items, text_len   # las mismas funciones que usa el proyecto

full = pd.read_csv(RAW / "steam_games.csv", low_memory=False)

COLS_USADAS = ["app_id", "name", "release_date", "price", "price_status",
               "developers", "publishers", "genres", "categories", "tags",
               "positive", "negative", "achievements", "supported_languages",
               "windows", "mac", "linux"]

lite = full[COLS_USADAS].copy()
lite["n_screenshots"] = full["screenshots"].map(count_items)
lite["n_movies"] = full["movies"].map(count_items)
lite["len_short_desc"] = full["short_description"].map(text_len)
lite["len_detailed_desc"] = full["detailed_description"].map(text_len)

OUT = RAW / "steam_games_lite.csv.gz"
lite.to_csv(OUT, index=False, compression="gzip")

mb = lambda p: p.stat().st_size / 1024**2
print(f"Original: {mb(RAW / 'steam_games.csv'):,.0f} MB | {full.shape}")
print(f"Ligero:   {mb(OUT):,.1f} MB | {lite.shape}")

Original: 1,097 MB | (150863, 42)
Ligero:   13.4 MB | (150863, 21)
